# Python: Почему [] в аргументах функции — это ловушка?

Link 
https://habr.com/ru/articles/982534/

In [1]:
def add_item(item, storage=[]):
    storage.append(item)
    return storage

print(add_item("яблоко"))  # Вывод: ['яблоко']
print(add_item("банан"))   # Вывод: ['яблоко', 'банан']
print(add_item("груша"))   # Вывод: ['яблоко', 'банан', 'груша']

['яблоко']
['яблоко', 'банан']
['яблоко', 'банан', 'груша']


Вместо того чтобы каждый раз начинать с чистого листа, функция начинает «запоминать» результаты предыдущих операций. Создается иллюзия, что у функции появилась скрытая память или статическая переменная, которая хранит состояние между вызовами.

### Корень проблемы: Время определения vs Время выполнения
Чтобы понять, почему список «запоминает» значения, нужно разобраться в том, как Python обрабатывает инструкцию def.

В большинстве языков программирования аргументы по умолчанию вычисляются каждый раз в момент вызова функции. В Python всё иначе: значения по умолчанию вычисляются ровно один раз — в момент определения функции (Definition Time).

Как это работает «под капотом»:
Когда интерпретатор читает файл и встречает def func(arg=[]), он выполняет это выражение.

Он создает объект пустого списка в памяти.

Он сохраняет ссылку на этот конкретный объект внутри самой функции — в скрытом атрибуте __defaults__.

Когда вы вызываете функцию без аргумента, Python не создает новый список. Он просто берет уже существующий объект из «кармана» __defaults__.



In [2]:
def add_item(item, storage=[]):
    storage.append(item)

# Сразу после создания функции:
print(add_item.__defaults__)  # Вывод: ([],) — список уже здесь!

add_item("X")

# После одного вызова:
print(add_item.__defaults__)  # Вывод: (['X'],) — тот же самый объект изменился

([],)
(['X'],)


### Каноничное решение: Идиома None

Чтобы избежать нежелательного сохранения состояния, в сообществе Python выработался стандарт (идиома), который считается единственно верным способом инициализации изменяемых аргументов.

Реализация паттерна
Вместо того чтобы создавать список в момент определения функции, мы используем None в качестве «заглушки». Создание же реального объекта переносится непосредственно в тело функции — туда, где код выполняется при каждом вызове.

In [ ]:
def add_item(item, storage=None):
    if storage is None:
        storage = []
    storage.append(item)
    return storage